# TF-IDF Vectorization - Avi Paudel

This notebook turns the labeled contract chunks into numbers a model can learn from.

1. Loads the labeled chunks from `Labeling.ipynb`, which uses the chunks from `Subword_Tokenization_Chunking.ipynb`
2. Vectorizes each chunk's text with **TF-IDF**, using the word tokenizer from `Tokenization.ipynb`
3. Checks that the vectors make sense
4. Builds the label matrix `y` (41 yes/no columns per chunk)

At the end we have:
- `X_train`, `X_test`: one row of TF-IDF numbers per chunk (the **input**)
- `y_train`, `y_test`: one row of 41 zeros and ones per chunk (the **answers**)

These are exactly what the TF-IDF baseline model needs.

In [1]:
import json, os, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Load the labeled chunks

The files are in `data/processed/`. Each row is one chunk of a contract (up to 512 RoBERTa tokens, 128-token overlap). The columns are `title`, `chunk_id`, `char_start`, `char_end`, `chunk_text`, then one 0/1 column for each of the 41 clause categories.

In [2]:
DATA_DIR = '../data/processed'

if not os.path.exists(DATA_DIR):
    try:
        # Google Colab: upload the 3 files from data/processed/
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"{DATA_DIR} not found. Run `git merge origin/main` to get the labeled chunks from Labeling.ipynb.")
    files.upload()
    DATA_DIR = '.'

In [3]:
train_df = pd.read_parquet(f'{DATA_DIR}/train_chunks_labeled.parquet')
test_df = pd.read_parquet(f'{DATA_DIR}/test_chunks_labeled.parquet')
with open(f'{DATA_DIR}/categories.json', 'r') as file:
    CATEGORIES = json.load(file)

print("Train chunks:", train_df.shape)   # expect (14181, 46)
print("Test chunks: ", test_df.shape)    # expect (2898, 46)
print("Categories:  ", len(CATEGORIES))  # expect 41

Train chunks: (14181, 46)
Test chunks:  (2898, 46)
Categories:   41


In [4]:
train_df[['title', 'chunk_id', 'chunk_text'] + CATEGORIES[:3]].head(3)

,title,chunk_id,chunk_text,Affiliate License-Licensee,Affiliate License-Licensor,Agreement Date
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,EXHIBIT 10.6\n\n ...,0,0,1
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,1,produced and supplied as needed pursuant to th...,0,0,0
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,2,"Market, and Distributor hereby accepts...",0,0,0


## 2. Tokenizer

TF-IDF first needs to split each chunk into words. I reuse `tokenize_words` from `Tokenization.ipynb` so every notebook splits text the same way:
- lowercase, punctuation dropped
- hyphenated and possessive words stay whole (`non-compete`, `party's`)
- dollar amounts become `<money>`, other numbers become `<num>`
- no stopwords removed (words like *not*, *shall*, *may* change the meaning of a clause)

In [5]:
TOKEN_PATTERN = r"\$\s*\d+(?:[.,]\d+)*|\d+(?:[.,]\d+)*|[a-z]+(?:[-'][a-z]+)*"

def tokenize_words(text):
    tokens = re.findall(TOKEN_PATTERN, text.lower())
    cleaned = []
    for token in tokens:
        if token.startswith("$"):
            cleaned.append("<money>")
        elif token[0].isdigit():
            cleaned.append("<num>")
        else:
            cleaned.append(token)
    return cleaned

In [6]:
tokenize_words("The Licensee shall NOT assign this Non-Compete Agreement and shall pay $1,000,000 within 30 days.")

['the',
 'licensee',
 'shall',
 'not',
 'assign',
 'this',
 'non-compete',
 'agreement',
 'and',
 'shall',
 'pay',
 '<money>',
 'within',
 '<num>',
 'days']

## 3. TF-IDF vectorization

**What vectorization means:** a model can only do math on numbers, so each chunk of text has to become a list of numbers. Every list has the same length (one spot per word in the vocabulary), and each spot always means the same word.

**What TF-IDF puts in each spot:**
- **TF (term frequency):** how often the word appears in *this* chunk
- **IDF (inverse document frequency):** how *rare* the word is across all chunks

Words that are in almost every chunk (*the*, *agreement*, *shall*) get a small weight. Words that point to a specific clause (*assign*, *indemnify*, *escrow*) get a bigger weight.

**Settings:**
- `tokenizer=tokenize_words`: use the tokenizer above
- `ngram_range=(1, 2)`: use single words **and** two-word phrases, e.g. "not assign", "governing law"
- `min_df=5`: ignore words/phrases found in fewer than 5 chunks (typos, names, one-offs)
- `sublinear_tf=True`: a word appearing 20 times in a chunk doesn't count 20x more than appearing once
- **No `max_df`:** following the Tokenization notebook, we don't drop common words, because that would remove *not*, *shall* and *may*. IDF already gives common words a low weight.

In [7]:
vectorizer = TfidfVectorizer(
    tokenizer=tokenize_words,
    lowercase=False,        # tokenize_words already lowercases
    token_pattern=None,     # not used, since we give our own tokenizer
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True,
)

**Fit on train only.** `fit_transform` learns the vocabulary and IDF weights from the **training** chunks and vectorizes them. For the test chunks we only call `transform`, which reuses what was learned from train. If the test set helped build the vocabulary, our test scores would look better than they really are (data leakage).

In [8]:
X_train = vectorizer.fit_transform(train_df['chunk_text'])
X_test = vectorizer.transform(test_df['chunk_text'])

print("X_train:", X_train.shape)   # (number of chunks, number of words/phrases)
print("X_test: ", X_test.shape)

X_train: (14181, 98157)
X_test:  (2898, 98157)


## 4. Checking the vectors

In [9]:
vocab = vectorizer.get_feature_names_out()
print("Vocabulary size:", len(vocab))

# Most of each row is 0, because a chunk only uses a small part of the vocabulary
filled = X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100
print(f"Non-zero values: {filled:.2f}% of the matrix")
print(f"Average words/phrases per chunk: {X_train.nnz / X_train.shape[0]:.0f}")

Vocabulary size: 98157
Non-zero values: 0.34% of the matrix
Average words/phrases per chunk: 332


The protected words from the Tokenization notebook should all still be in the vocabulary:

In [10]:
protected = ['not', 'no', 'without', 'shall', 'must', 'may', 'only', 'except', 'unless', '<money>', '<num>', 'shall not', 'not assign']
for word in protected:
    print(f"{word:12s} in vocabulary: {word in vectorizer.vocabulary_}")

not          in vocabulary: True
no           in vocabulary: True
without      in vocabulary: True
shall        in vocabulary: True
must         in vocabulary: True
may          in vocabulary: True
only         in vocabulary: True
except       in vocabulary: True
unless       in vocabulary: True
<money>      in vocabulary: True
<num>        in vocabulary: True
shall not    in vocabulary: True
not assign   in vocabulary: True


**IDF weights:** the most common words should get the lowest weight. The rarest ones are names and odd phrases that only appear in a handful of chunks.

In [11]:
idf = pd.Series(vectorizer.idf_, index=vocab).sort_values()
print("Lowest IDF (most common):\n", idf.head(10).round(2).to_string())
print("\nHighest IDF (rarest):\n", idf.tail(10).round(2).to_string())

Lowest IDF (most common):
 <num>     1.03
of        1.04
and       1.04
the       1.04
to        1.05
in        1.08
a         1.11
or        1.13
by        1.14
of the    1.16

Highest IDF (rarest):
 overriding       8.77
overseen by      8.77
oversight and    8.77
owed but         8.77
owed ibm         8.77
zoning or        8.77
<money> all      8.77
zynga and        8.77
<money> a        8.77
zynga game       8.77


**One sentence as a vector:** the vectorizer turns any text into a row of the same length as the vocabulary. Almost every spot is 0; only the words in the sentence get a number.

In [12]:
sentence = "This Agreement shall be governed by the laws of the State of New York."
vector = vectorizer.transform([sentence]).toarray()[0]

print("Length of the vector:", len(vector))
print("Non-zero spots:      ", (vector > 0).sum())

top = vector.argsort()[::-1][:10]
pd.Series(vector[top], index=vocab[top]).round(3)

Length of the vector: 98157
Non-zero spots:       25


of new         0.320
new york       0.292
york           0.291
be governed    0.283
governed by    0.279
governed       0.269
state of       0.264
the state      0.263
the laws       0.244
laws of        0.238
dtype: float64

**Top words per category:** for each category, I compare the average TF-IDF vector of its chunks (labeled 1) with the average over all chunks. The words that score much higher in that category's chunks should match the clause type. This is also a preview of what the baseline model will learn.

In [13]:
average_all = np.asarray(X_train.mean(axis=0))[0]

def category_top_words(category, n=8):
    rows = (train_df[category] == 1).to_numpy()
    average_category = np.asarray(X_train[rows].mean(axis=0))[0]
    difference = average_category - average_all
    top = difference.argsort()[::-1][:n]
    return ', '.join(vocab[top])

for category in ['Governing Law', 'Anti-Assignment', 'Non-Compete', 'Insurance',
                 'Cap On Liability', 'Termination For Convenience', 'Source Code Escrow']:
    print(f"{category:28s} -> {category_top_words(category)}")

Governing Law                -> the state, governed, state of, laws of, be governed, the laws, governed by, laws
Anti-Assignment              -> assign, assignment, consent, written consent, consent of, without the, prior written, party
Non-Compete                  -> competitive, products, or indirectly, indirectly, directly, you, directly or, during the
Insurance                    -> insurance, liability insurance, liability, coverage, such insurance, of insurance, maintain, policy
Cap On Liability             -> damages, liability, consequential, liable, be liable, claim, arising, incidental
Termination For Convenience  -> termination, terminate, terminate this, written notice, notice, may terminate, <num> termination, written
Source Code Escrow           -> escrow, source code, the escrow, the source, escrow agreement, code, ncc, escrow agent


## 5. Label matrix `y`

The labels were already built in `Labeling.ipynb`: a chunk has a **1** for a category if a lawyer-highlighted answer of that category overlaps it, otherwise **0**. Here I just take those 41 columns out as a matrix, in the same order as `categories.json`.

Row *i* of `X_train` and row *i* of `y_train` describe the same chunk, because both come from row *i* of `train_df`.

In [14]:
y_train = train_df[CATEGORIES].to_numpy()
y_test = test_df[CATEGORIES].to_numpy()

print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

# X and y must have the same number of rows (one per chunk)
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
print("X and y line up.")

y_train: (14181, 41)
y_test:  (2898, 41)
X and y line up.


In [15]:
counts = pd.DataFrame({
    'train_positive_chunks': y_train.sum(axis=0),
    'test_positive_chunks': y_test.sum(axis=0),
}, index=CATEGORIES).sort_values('train_positive_chunks')

counts.head(10)   # rarest categories

,train_positive_chunks,test_positive_chunks
Unlimited/All-You-Can-Eat-License,31,5
Price Restrictions,34,0
Third Party Beneficiary,42,15
Source Code Escrow,47,2
Most Favored Nation,51,5
No-Solicit Of Customers,53,18
Affiliate License-Licensor,63,22
Non-Disparagement,64,15
No-Solicit Of Employees,86,19
Joint Ip Ownership,99,15


Some categories have very few positive chunks, and **Price Restrictions has 0 in test**, so its F1 can't be computed. The baseline should use per-category precision/recall/F1 and `class_weight="balanced"`.

## 6. Summary

**What this notebook does:** turns each contract chunk into a TF-IDF vector (`X`) and pulls out the 41-column label matrix (`y`), ready for the TF-IDF baseline model.

**Results:**
- `X_train`: 14,181 × 98,157, `X_test`: 2,898 × 98,157 (chunks × words/phrases)
- `y_train`: 14,181 × 41, `y_test`: 2,898 × 41 (chunks × categories)
- Vocabulary: 98,157 words and two-word phrases; each chunk uses about 332 of them (0.34% of the matrix is non-zero), so the matrix is mostly zeros (sparse)
- Protected words (*not*, *shall*, *may*, ...) are all kept
- Top words per category match the clause type (Governing Law → *governed by*, *laws of*; Anti-Assignment → *assign*, *written consent*; Insurance → *insurance*, *coverage*), so the vectors carry useful information